In [1]:
%load_ext sql

In [2]:
%sql sqlite:///Northwind.db

'Connected: @Northwind.db'

In [3]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

### Describe the Structure of the Database

In [4]:
%%sql

SELECT
    name, sql
FROM
    sqlite_master
WHERE
    type = 'table'

 * sqlite:///Northwind.db
Done.


name,sql
Categories,"CREATE TABLE Categories ( CategoryID int(11) NOT NULL, CategoryName varchar(15) NOT NULL, Description mediumtext, Picture longblob, PRIMARY KEY (CategoryID))"
Customers,"CREATE TABLE Customers ( CustomerID varchar(5) NOT NULL, CompanyName varchar(40) NOT NULL, ContactName varchar(30) DEFAULT NULL, ContactTitle varchar(30) DEFAULT NULL, Address varchar(60) DEFAULT NULL, City varchar(15) DEFAULT NULL, Region varchar(15) DEFAULT NULL, PostalCode varchar(10) DEFAULT NULL, Country varchar(15) DEFAULT NULL, Phone varchar(24) DEFAULT NULL, Fax varchar(24) DEFAULT NULL, PRIMARY KEY (CustomerID) )"
Region,"CREATE TABLE Region ( RegionID int(11) NOT NULL, RegionDescription varchar(50) NOT NULL, PRIMARY KEY (RegionID))"
Shippers,"CREATE TABLE Shippers ( ShipperID int(11) NOT NULL, CompanyName varchar(40) NOT NULL, Phone varchar(24) DEFAULT NULL, PRIMARY KEY (ShipperID))"
Suppliers,"CREATE TABLE Suppliers ( SupplierID int(11) NOT NULL, CompanyName varchar(40) NOT NULL, ContactName varchar(30) DEFAULT NULL, ContactTitle varchar(30) DEFAULT NULL, Address varchar(60) DEFAULT NULL, City varchar(15) DEFAULT NULL, Region varchar(15) DEFAULT NULL, PostalCode varchar(10) DEFAULT NULL, Country varchar(15) DEFAULT NULL, Phone varchar(24) DEFAULT NULL, Fax varchar(24) DEFAULT NULL, HomePage mediumtext, PRIMARY KEY (SupplierID))"
Employees,"CREATE TABLE Employees ( EmployeeID int(11) NOT NULL, LastName varchar(20) NOT NULL, FirstName varchar(10) NOT NULL, Title varchar(30) DEFAULT NULL, TitleOfCourtesy varchar(25) DEFAULT NULL, BirthDate datetime DEFAULT NULL, HireDate datetime DEFAULT NULL, Address varchar(60) DEFAULT NULL, City varchar(15) DEFAULT NULL, Region varchar(15) DEFAULT NULL, PostalCode varchar(10) DEFAULT NULL, Country varchar(15) DEFAULT NULL, HomePhone varchar(24) DEFAULT NULL, Extension varchar(4) DEFAULT NULL, Notes mediumtext NOT NULL, ReportsTo int(11) DEFAULT NULL, PhotoPath varchar(255) DEFAULT NULL, Salary float DEFAULT NULL, PRIMARY KEY (EmployeeID),CONSTRAINT FK_Employees_Employees FOREIGN KEY (ReportsTo) REFERENCES Employees (EmployeeID) )"
EmployeeTerritories,"CREATE TABLE EmployeeTerritories ( EmployeeID int(11) NOT NULL, TerritoryID varchar(20) NOT NULL, PRIMARY KEY (EmployeeID,TerritoryID), CONSTRAINT FK_EmployeeTerritories_Territories FOREIGN KEY (TerritoryID) REFERENCES Territories (TerritoryID), CONSTRAINT FK_EmployeeTerritories_Employees FOREIGN KEY (EmployeeID) REFERENCES Employees (EmployeeID))"
OrderDetails,"CREATE TABLE OrderDetails ( OrderID int(11) NOT NULL, ProductID int(11) NOT NULL, UnitPrice decimal(10,4) NOT NULL DEFAULT 0.0000, Quantity smallint(2) NOT NULL DEFAULT 1, Discount double(8,0) NOT NULL DEFAULT 0, PRIMARY KEY (OrderID,ProductID), CONSTRAINT FK_Order_Details_Products FOREIGN KEY (ProductID) REFERENCES Products (ProductID), CONSTRAINT FK_Order_Details_Orders FOREIGN KEY (OrderID) REFERENCES Orders (OrderID))"
Orders,"CREATE TABLE Orders ( OrderID int(11) NOT NULL, CustomerID varchar(5) DEFAULT NULL, EmployeeID int(11) DEFAULT NULL, OrderDate datetime DEFAULT NULL, RequiredDate datetime DEFAULT NULL, ShippedDate datetime DEFAULT NULL, ShipVia int(11) DEFAULT NULL, Freight decimal(10,4) DEFAULT '0.0000', ShipName varchar(40) DEFAULT NULL, ShipAddress varchar(60) DEFAULT NULL, ShipCity varchar(15) DEFAULT NULL, ShipRegion varchar(15) DEFAULT NULL, ShipPostalCode varchar(10) DEFAULT NULL, ShipCountry varchar(15) DEFAULT NULL, PRIMARY KEY (OrderID), CONSTRAINT FK_Orders_Shippers FOREIGN KEY (ShipVia) REFERENCES Shippers (ShipperID), CONSTRAINT FK_Orders_Customers FOREIGN KEY (CustomerID) REFERENCES Customers (CustomerID), CONSTRAINT FK_Orders_Employees FOREIGN KEY (EmployeeID) REFERENCES Employees (EmployeeID))"
Territories,"CREATE TABLE Territories ( TerritoryID varchar(20) NOT NULL, TerritoryDescription varchar(50) NOT NULL, RegionID int(11) NOT NULL, PRIMARY KEY (TerritoryID), CONSTRAINT FK_Territories_Region FOREIGN KEY (RegionID) REFERENCES Region (RegionID))"


### Task One: Ranking all the orders of a specific customer from the most recent to the least recent using window functions

In [7]:
%%sql

SELECT
    *,
    RANK() OVER (ORDER BY ORDERDATE DESC) AS ORDER_DATE
FROM
    orders
WHERE
    customerId = 'ALFKI';

 * sqlite:///Northwind.db
Done.


OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,ShipAddress,ShipCity,ShipRegion,ShipPostalCode,ShipCountry,ORDER_DATE
11011,ALFKI,3,1998-04-09 00:00:00,1998-05-07 00:00:00,1998-04-13 00:00:00,1,1.21,Alfred-s Futterkiste,Obere Str. 57,Berlin,None,12209,Germany,1
10952,ALFKI,1,1998-03-16 00:00:00,1998-04-27 00:00:00,1998-03-24 00:00:00,1,40.42,Alfred-s Futterkiste,Obere Str. 57,Berlin,None,12209,Germany,2
10835,ALFKI,1,1998-01-15 00:00:00,1998-02-12 00:00:00,1998-01-21 00:00:00,3,69.53,Alfred-s Futterkiste,Obere Str. 57,Berlin,None,12209,Germany,3
10702,ALFKI,4,1997-10-13 00:00:00,1997-11-24 00:00:00,1997-10-21 00:00:00,1,23.94,Alfred-s Futterkiste,Obere Str. 57,Berlin,None,12209,Germany,4
10692,ALFKI,4,1997-10-03 00:00:00,1997-10-31 00:00:00,1997-10-13 00:00:00,2,61.02,Alfred-s Futterkiste,Obere Str. 57,Berlin,None,12209,Germany,5
10643,ALFKI,6,1997-08-25 00:00:00,1997-09-22 00:00:00,1997-09-02 00:00:00,1,29.46,Alfreds Futterkiste,Obere Str. 57,Berlin,None,12209,Germany,6


### Task Two: Calculating a running total of the quantity of orders using window functions

In [12]:
%%sql

SELECT
    *,
    ROUND(SUM(UnitPrice * Quantity - Discount) OVER (PARTITION BY OrderID ORDER BY ProductID), 2) AS Orders_running_Total
FROM
    orderdetails;

 * sqlite:///Northwind.db
Done.


OrderID,ProductID,UnitPrice,Quantity,Discount,Orders_running_Total
10248,11,14,12,0.0,168.0
10248,42,9.8,10,0.0,266.0
10248,72,34.8,5,0.0,440.0
10249,14,18.6,9,0.0,167.4
10249,51,42.4,40,0.0,1863.4
10250,41,7.7,10,0.0,77.0
10250,51,42.4,35,0.0,1561.0
10250,65,16.8,15,0.0,1813.0
10251,22,16.8,6,0.0,100.8
10251,57,15.6,15,0.0,334.8


### Task Three: Using Window Functions to find the difference between successive order dates for each customer

In [14]:
%%sql

SELECT
    customerid,
    orderdate,
    LAG(orderdate) OVER (
        PARTITION BY customerid
        ORDER BY orderdate
    ) AS PrevOrderDate,
    julianday(orderdate) - julianday(LAG(orderdate) OVER (
        PARTITION BY customerid
        ORDER BY orderdate
    )) AS DateDiff
FROM
    orders;

 * sqlite:///Northwind.db
Done.


CustomerID,OrderDate,PrevOrderDate,DateDiff
ALFKI,1997-08-25 00:00:00,None,None
ALFKI,1997-10-03 00:00:00,1997-08-25 00:00:00,39.0
ALFKI,1997-10-13 00:00:00,1997-10-03 00:00:00,10.0
ALFKI,1998-01-15 00:00:00,1997-10-13 00:00:00,94.0
ALFKI,1998-03-16 00:00:00,1998-01-15 00:00:00,60.0
ALFKI,1998-04-09 00:00:00,1998-03-16 00:00:00,24.0
ANATR,1996-09-18 00:00:00,None,None
ANATR,1997-08-08 00:00:00,1996-09-18 00:00:00,324.0
ANATR,1997-11-28 00:00:00,1997-08-08 00:00:00,112.0
ANATR,1998-03-04 00:00:00,1997-11-28 00:00:00,96.0


### Task Four: Calculating the moving average of the quantity of the last 3 orders for each product using window functions

In [20]:
%%sql

SELECT
    orderid,
    productid,
    quantity,
    AVG(quantity) OVER (PARTITION BY productid ORDER BY orderid ROWS BETWEEN 2 PRECEDING AND 0 FOLLOWING) AS MovingAvgQuantity
FROM 
    OrderDetails
ORDER BY
    orderid;

 * sqlite:///Northwind.db
Done.


OrderID,ProductID,Quantity,MovingAvgQuantity
10248,11,12,12.0
10248,42,10,10.0
10248,72,5,5.0
10249,14,9,9.0
10249,51,40,40.0
10250,41,10,10.0
10250,51,35,37.5
10250,65,15,15.0
10251,22,6,6.0
10251,57,15,15.0
